Sums fluxes by feature in BEF supported polygons shapefile

Per Claude session 'GHG flux calculation from shapefiles'

In [1]:
import geopandas as gpd
import xarray as xr
import rioxarray  
import coiled
import pandas as pd
import dask
import sys
from pathlib import Path
from dask.distributed import Client
import numpy as np
import rasterio
import rasterio.features
from flox.xarray import xarray_reduce
import math
from shapely.geometry import box
from shapely import wkb as shapely_wkb
from shapely.validation import explain_validity, make_valid
from shapely.geometry import GeometryCollection, MultiPolygon, Polygon
import numpy as np
from affine import Affine
import dask.array as da
from datetime import date
from flox import ReindexArrayType, ReindexStrategy

# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

pd.set_option('display.max_columns', None)

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu
from src.utilities import zonal_stats_utilities as zsu
from src.utilities.create_cluster import create_cluster

In [2]:
### Run parameters

shapefile_path = "/mnt/c/GIS/shapefiles/BEF/BEF_supported_polygons_1x1_grid_clip.shp"   # WH site buffers

veg_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_vegetation/version_1_0_5__standard__global/mega_zarr/annual_intervals/4000_pixels/20260130/vegetation_zarr.zarr/"
organic_soil_zarr_path = "s3://gfw2=data/climate/AFOLU_flux_model/organic_soils/outputs/version_1_0_1/mega_zarr/ogh_mixed_f1_f15_f2_20260513/five_year/4000_pixels/20260525/mega.zarr/"  
min_soil_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_soil_organic_carbon/version_1_0_1__standard__global/zarr/4000_pixels/20260611/SOC_zarr.zarr/"  
lulucf_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_LULUCF_totals/LULUCF_version_1_0_0_standard__global__veg_v1_0_5__org_soil_v1_0_1__min_soil_v1_0_1/zarr/annual_intervals/4000_pixels/20260614/LULUCF_annual.zarr/"
years = cn.veg_outputs_years  # [2016..2024], indexed 0-8 in the zarr's year dimension

out_name = "BEF_suported_polygons"
readme_description = "GHG flux zonal statistics for BEF supported AOIs"

today = date.today().strftime("%Y-%m-%d")
today_sparse = date.today().strftime("%Y%m%d")

readme_lines = [
    "Zonal statistics from Global Nature Watch's Land GHG Monitoring System: vegetation and soil flux data only",
    "CAUTION: DATA ARE PRELIMINARY AND UNPUBLISHED; DO NOT DISTRIBUTE WITHOUT PRIOR APPROVAL",
    "Created by Melissa Rose",
    "melissa.rose@wri.org",
    today,
    readme_description,
    f"Shapefile used: {shapefile_path}",
    f"Years covered: {cn.veg_outputs_years[0]}-{cn.LC_last_year} ({cn.veg_end_year_count} years)",
    # f"Vegetation model: v{cn.veg_model_version}; time step: annual",
    f"Vegetation model: v1.0.5; time step: annual",  # Using this because the repo has moved on to v1.0.6 but I'm still using veg v1.0.5 for this analysis
    f"Organic soil model: v{cn.organic_soil_model_version}; time step: 5 years (2016-2020, 2021-2024)",
    f"Mineral soil model: v{cn.SOC_model_version}; time step: static (same value all years)",
    f"LULUCF model: v{cn.LULUCF_model_version}; time step: annual, with soil datasets annualized to match vegetation",
    f"Vegetation zarr: {veg_zarr_path}",
    f"Organic soil zarr: {organic_soil_zarr_path}",
    f"Mineral soil zarr: {min_soil_zarr_path}",
    f"LULUCF zarr: {lulucf_zarr_path}",
    "",
    "Emissions/net sources are positive; removals/net sinks are negative.",
    "land_state_node, land_state_meaning, and derived columns are just for vegetation data; they do not apply to soil or LULUCF data",
    "primary_forest is as defined in LULUCF manuscript; it is a combination of humid tropical primary forest, intact forest landscapes outside the tropics, and forest >100 years old.",
    "Calculated using flox zonal stats in a Coiled cluster" 
] 

### Shapefile processing

In [4]:
gdf = gpd.read_file(shapefile_path, encoding="utf-8")
print("Original CRS:", gdf.crs)

if gdf.crs is None:
    raise ValueError("Shapefile has no CRS defined (missing/invalid .prj) -- can't safely reproject.")
elif gdf.crs.to_epsg() != 4326:
    print(f"Reprojecting from {gdf.crs.to_epsg()} to EPSG:4326")
    gdf = gdf.to_crs(epsg=4326)
else:
    print("Already EPSG:4326 -- no reprojection needed")

print(gdf.crs, len(gdf), "features")
print(gdf.columns)


# # Shapefile columns to carry through to the output table

# # UNESCO World Heritage sites 2021
# field = "REF"  # unique feature ID used for the zonal grouping -- must be castable to an integer
# output_fields = ["NAME", "NAME_FR", "INSC_DATE", "CATEGORY", "REGION"]   # Additional fields to carry through

# World Heritage buffers
field = "ID"
output_fields = ["Country", "STD_NAME", "WDPAID", "Lead_organ", "Type_of_In", "POL_ID"]

print("Checking field names")
missing = [f for f in [field] + output_fields if f not in gdf.columns]
if missing:
    raise ValueError(f"Field(s) not found in shapefile: {missing}. Available columns: {list(gdf.columns)}")
print("Field names valid")

Original CRS: EPSG:4326
Already EPSG:4326 -- no reprojection needed
EPSG:4326 2059 features
Index(['Country', 'STD_NAME', 'WDPAID', 'Lead_organ', 'Type_of_In', 'POL_ID',
       'ID', 'geometry'],
      dtype='object')
Checking field names
Field names valid


In [5]:
# Identifies the shapefiles features that intersect each tile

minx, miny, maxx, maxy = gdf.total_bounds
west_edges = range(int(math.floor(minx / 10) * 10), int(math.ceil(maxx / 10) * 10), 10)
north_edges = range(int(math.floor(miny / 10) * 10) + 10, int(math.ceil(maxy / 10) * 10) + 10, 10)
candidate_tile_ids = [uu.xy_to_tile_id(west, north) for west in west_edges for north in north_edges]

features_by_tile = {}
for tile_id in candidate_tile_ids:
    tminx, tminy, tmaxx, tmaxy = uu.get_10x10_tile_bounds(tile_id)
    subset = gdf[gdf.geometry.intersects(box(tminx, tminy, tmaxx, tmaxy))]
    if len(subset):
        features_by_tile[tile_id] = subset

print({tid: len(sub) for tid, sub in features_by_tile.items()})

{'10S_180W': 10, '20S_170W': 1, '10S_170W': 2, '20S_160W': 2, '10S_160W': 5, '20S_150W': 2, '10S_150W': 22, '00N_150W': 5, '20S_140W': 3, '10S_140W': 2, '00N_140W': 3, '60N_140W': 18, '70N_140W': 18, '60N_130W': 2, '70N_130W': 3, '70N_120W': 9, '80N_120W': 1, '20N_110W': 1, '70N_110W': 3, '80N_110W': 3, '00N_100W': 1, '10N_100W': 2, '80N_100W': 4, '00N_090W': 16, '10N_090W': 4, '20N_090W': 2, '60N_090W': 1, '80N_090W': 5, '10S_080W': 101, '00N_080W': 325, '10N_080W': 203, '20N_080W': 14, '60N_080W': 1, '80N_080W': 6, '20S_070W': 16, '10S_070W': 126, '00N_070W': 22, '10N_070W': 24, '70N_070W': 3, '10S_060W': 16, '70N_060W': 1, '00N_000E': 36, '10N_000E': 86, '00N_010E': 395, '10N_010E': 229, '00N_020E': 74, '10N_020E': 92, '10N_130E': 48, '20N_140E': 20, '30N_140E': 1, '20S_150E': 1, '10S_150E': 1, '00N_150E': 49, '20S_160E': 35, '10S_160E': 4, '10N_160E': 6, '20N_160E': 5, '20S_170E': 1, '10S_170E': 95, '00N_170E': 12, '10N_170E': 3}


In [6]:
# Checks validity of features being processed

for tile in features_by_tile:
    print("tile_id:", tile)
    subset = features_by_tile[tile]
    print(field, subset[field].tolist())
    print("is_valid:", subset.geometry.is_valid.tolist())
    print("is_empty:", subset.geometry.is_empty.tolist())
    print("geom_type:", subset.geometry.geom_type.tolist())
    print("bounds:\n", subset.geometry.bounds)

    for _, row in subset[~subset.geometry.is_valid].iterrows():
        print(f"{field} {row[field]}: {explain_validity(row.geometry)}")

    print("\n")

tile_id: 10S_180W
ID [0, 1, 2, 3, 4, 5, 37, 38, 2036, 2037]
is_valid: [True, True, True, True, True, True, True, True, True, True]
is_empty: [False, False, False, False, False, False, False, False, False, False]
geom_type: ['Polygon', 'Polygon', 'Polygon', 'Polygon', 'Polygon', 'Polygon', 'MultiPolygon', 'MultiPolygon', 'MultiPolygon', 'MultiPolygon']
bounds:
             minx       miny        maxx       maxy
0    -179.961274 -16.852717 -179.765094 -16.618746
1    -178.960516 -17.294375 -178.893941 -17.246415
2    -179.024806 -17.376951 -178.853030 -17.248285
3    -178.997284 -17.351693 -178.975355 -17.308575
4    -179.039280 -17.258506 -178.965142 -17.227082
5    -179.071616 -17.227255 -178.962493 -17.133119
37   -171.912867 -14.083711 -171.390910 -13.814356
38   -171.877182 -14.018638 -171.802052 -13.985368
2036 -179.999762 -16.332665  179.999763 -16.100780
2037 -179.999885 -17.012954  179.999887 -16.867763


tile_id: 20S_170W
ID [7]
is_valid: [True]
is_empty: [False]
geom_type: ['P

In [7]:
# Fixes feature geometries and rebuilds feature lists for each tile for processing

def repair(geom):
    if geom.is_valid:
        return geom
    fixed = make_valid(geom)
    if isinstance(fixed, GeometryCollection):
        print(f"Fixing {GeometryCollection}")
        polys = [g for g in fixed.geoms if isinstance(g, (Polygon, MultiPolygon))]
        fixed = MultiPolygon(polys) if len(polys) > 1 else (polys[0] if polys else fixed)
    return fixed

print("Fixing geometries")
areas_before = gdf.geometry.area.copy()
gdf["geometry"] = gdf.geometry.apply(repair)
areas_after = gdf.geometry.area

rel_diff = (areas_after - areas_before).abs() / areas_before.replace(0, np.nan)

print("All valid now:", gdf.geometry.is_valid.all())
print("Max relative area change:", rel_diff.max())

changed = rel_diff[rel_diff > 1e-6]
if len(changed):
    print("Features whose area changed by more than a negligible amount:")
    print(gdf.loc[changed.index, [field]].assign(rel_area_change=changed))
else:
    print("No feature's area changed by more than a negligible amount -- safe to proceed.")

# Rebuilds features_by_tile from the repaired gdf -- the previous version was built from the
# pre-repair geometries, so it would still hold the invalid ones otherwise.
minx, miny, maxx, maxy = gdf.total_bounds
west_edges = range(int(math.floor(minx / 10) * 10), int(math.ceil(maxx / 10) * 10), 10)
north_edges = range(int(math.floor(miny / 10) * 10) + 10, int(math.ceil(maxy / 10) * 10) + 10, 10)
candidate_tile_ids = [uu.xy_to_tile_id(west, north) for west in west_edges for north in north_edges]

features_by_tile = {}
for tile_id in candidate_tile_ids:
    tminx, tminy, tmaxx, tmaxy = uu.get_10x10_tile_bounds(tile_id)
    subset = gdf[gdf.geometry.intersects(box(tminx, tminy, tmaxx, tmaxy))]
    if len(subset):
        features_by_tile[tile_id] = subset

print({tid: len(sub) for tid, sub in features_by_tile.items()})
print("All valid across rebuilt features_by_tile:", all(sub.geometry.is_valid.all() for sub in features_by_tile.values()))

for tile_id, subset in features_by_tile.items():
    print(tile_id, subset.geometry.is_valid.tolist())

Fixing geometries


/tmp/ipykernel_5563/2738495339.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  areas_before = gdf.geometry.area.copy()
/tmp/ipykernel_5563/2738495339.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  areas_after = gdf.geometry.area


All valid now: True
Max relative area change: 1.672906512995536e-14
No feature's area changed by more than a negligible amount -- safe to proceed.
{'10S_180W': 10, '20S_170W': 1, '10S_170W': 2, '20S_160W': 2, '10S_160W': 5, '20S_150W': 2, '10S_150W': 22, '00N_150W': 5, '20S_140W': 3, '10S_140W': 2, '00N_140W': 3, '60N_140W': 18, '70N_140W': 18, '60N_130W': 2, '70N_130W': 3, '70N_120W': 9, '80N_120W': 1, '20N_110W': 1, '70N_110W': 3, '80N_110W': 3, '00N_100W': 1, '10N_100W': 2, '80N_100W': 4, '00N_090W': 16, '10N_090W': 4, '20N_090W': 2, '60N_090W': 1, '80N_090W': 5, '10S_080W': 101, '00N_080W': 325, '10N_080W': 203, '20N_080W': 14, '60N_080W': 1, '80N_080W': 6, '20S_070W': 16, '10S_070W': 126, '00N_070W': 22, '10N_070W': 24, '70N_070W': 3, '10S_060W': 16, '70N_060W': 1, '00N_000E': 36, '10N_000E': 86, '00N_010E': 395, '10N_010E': 229, '00N_020E': 74, '10N_020E': 92, '10N_130E': 48, '20N_140E': 20, '30N_140E': 1, '20S_150E': 1, '10S_150E': 1, '00N_150E': 49, '20S_160E': 35, '10S_160E': 

### Analysis and contextual layer inputs

In [8]:
### Analysis layers

# Mineral soil (uses change for 2020 only (2010-2015 vs. 2015-2020))
min_soil_year_idx = cn.SOC_change_intervals.index(2020) + 1
min_soil_var_names = {
    "mineral_soil_gross_loss_MgCO2": f"{cn.SOC_loss_min_soil_extent_pattern}_ha_yr",
    "mineral_soil_gross_gain_MgCO2": f"{cn.SOC_gain_min_soil_extent_pattern}_ha_yr",
    "mineral_soil_net_change_MgCO2": f"{cn.SOC_net_min_soil_extent_pattern}_ha_yr",
}
min_soil_ds = xr.open_zarr(min_soil_zarr_path, consolidated=False)
min_soil_layers_static = {
    label: min_soil_ds[var].isel(year=min_soil_year_idx).drop_vars("year")
        .rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
    for label, var in min_soil_var_names.items()
}

# Organic soil (five-year intervals: 2016_2020 and 2021_2024 -- kept at its own native two-interval
# resolution rather than collapsed to one year like mineral soil, since it has two blocks to preserve)
organic_soil_var_names = {
    "organic_soil_emissions_fire_MgCO2e": cn.burned_organic_soils_total_pattern,
    "organic_soil_emissions_drainage_MgCO2e": cn.drained_organic_soils_total_pattern,
}
organic_soil_ds = xr.open_zarr(cn.organic_soil_zarr_path, consolidated=False)

# organic_soil_ds actually has 5 native intervals (2001_2005, 2006_2010, 2011_2015, 2016_2020, 2021_2024) --
# only the last two (matching cn.organic_soil_year_intervals) are relevant to this model's 2016-2024 range.
# Takes the last N slices (N = len(cn.organic_soil_year_intervals)) rather than hardcoding indices 3/4,
# so this doesn't silently break if the zarr's historical coverage changes again later.
organic_soil_layers_native = {
    label: organic_soil_ds[var].isel(year=slice(-len(cn.organic_soil_year_intervals), None))
        .assign_coords(year=np.arange(len(cn.organic_soil_year_intervals)))
        .rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
    for label, var in organic_soil_var_names.items()
}

# Vegetation (annual)
veg_var_names = {
    # v1.0.6 names
    # "veg_gross_emissions_MgCO2e": f"{cn.gross_emis_all_C_pools_all_gases_pattern}_ha_yr",
    # "veg_gross_removals_MgCO2e": f"{cn.gross_removals_all_C_pools_pattern}_ha_yr",
    # "veg_net_flux_MgCO2e": f"{cn.net_flux_all_C_pools_all_gases_pattern}_ha_yr",

    # v1.0.5 names
    "veg_gross_emissions_MgCO2e": f"gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr",
    "veg_gross_removals_MgCO2e": f"gross_removals__all_C_pools__MgCO2_ha_yr",
    "veg_net_flux_MgCO2e": f"net_flux__all_C_pools__all_gases__MgCO2e_ha_yr",
}
veg_ds = xr.open_zarr(veg_zarr_path, consolidated=False)
veg_analysis_layers = {
    label: veg_ds[var].rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
    for label, var in veg_var_names.items()
}

# LULUCF (annual)
lulucf_var_names = {
    "LULUCF_gross_emissions_MgCO2e": f"{cn.gross_emis_all_C_pools_all_gases_LULUCF_pattern}_ha_yr",
    "LULUCF_gross_removals_MgCO2e": f"{cn.gross_removals_all_C_pools_LULUCF_pattern}_ha_yr",
    "LULUCF_net_flux_MgCO2e": f"{cn.net_flux_all_C_pools_all_gases_LULUCF_pattern}_ha_yr",
}
lulucf_ds = xr.open_zarr(lulucf_zarr_path, consolidated=False)
lulucf_layers = {
    label: lulucf_ds[var].rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
    for label, var in lulucf_var_names.items()
}

# Annual datasets merged-- merges directly into the same stack/reduce, unlike SOC.
annual_analysis_layers = {**veg_analysis_layers, **lulucf_layers}
var_names = {**veg_var_names, **lulucf_var_names, **min_soil_var_names, **organic_soil_var_names}

# Pixel area
pixel_area_ds = xr.open_zarr(cn.pixel_area_zarr_path, consolidated=False).rename_vars(band_data="pixel_area_ha")
pixel_area = pixel_area_ds["pixel_area_ha"].rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")

# Resolution and dimensions of the actual analysis-ready layers (not the raw zarrs) -- this correctly
# shows mineral soil with no year dim (collapsed to one value) and organic soil with year=2 (sliced
# down from the raw zarr's 5), rather than the raw zarrs' native shapes
min_soil_check = next(iter(min_soil_layers_static.values()))
organic_soil_check = next(iter(organic_soil_layers_native.values()))
veg_check = next(iter(veg_analysis_layers.values()))
lulucf_check = next(iter(lulucf_layers.values()))

print("mineral soil:  x res:", float(min_soil_check["x"][1] - min_soil_check["x"][0]), " size:", min_soil_check.sizes)
print("organic soil:  x res:", float(organic_soil_check["x"][1] - organic_soil_check["x"][0]), " size:", organic_soil_check.sizes)
print("vegetation  :  x res:", float(veg_check["x"][1] - veg_check["x"][0]), " size:", veg_check.sizes)
print("lulucf      :  x res:", float(lulucf_check["x"][1] - lulucf_check["x"][0]), " size:", lulucf_check.sizes)
print("pixel_area  :  x res:", float(pixel_area["x"][1] - pixel_area["x"][0]), " size:", pixel_area.sizes)

mineral soil:  x res: 0.0002499999999940883  size: Frozen({'y': 720000, 'x': 1440000})
organic soil:  x res: 0.0002499999999940883  size: Frozen({'year': 2, 'y': 720000, 'x': 1440000})
vegetation  :  x res: 0.0002499999999940883  size: Frozen({'year': 9, 'y': 720000, 'x': 1440000})
lulucf      :  x res: 0.0002499999999940883  size: Frozen({'year': 9, 'y': 720000, 'x': 1440000})
pixel_area  :  x res: 0.0002499999999940883  size: Frozen({'y': 560000, 'x': 1440000})


In [9]:
### Contextual layers

# land_state_node already lives inside the vegetation mega_zarr and has the same 9-year dimension
# as the flux variables, so no special year-alignment handling is needed (unlike SOC).
# land_state_da = veg_ds[cn.land_state_pattern].rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")  # v1.0.6 pattern
land_state_da = veg_ds['land_state_node'].rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")  # v1.0.5 pattern

state_node_df = zsu.create_state_node_df(cn.state_node_lookup_table_local, cn.state_node_lookup_table_s3, cn.sheet)
node_codes = np.array(list(state_node_df["land_state"]), dtype=np.uint32)

# starting_composite_primary_forest is a separate, static (single 2015 snapshot) zarr -- no year dimension
primary_forest_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/starting_composite_primary_forest/version_1_0_5__standard__global/2015/presence/zarr/4000_pixels/20260210/starting_composite_primary_forest.zarr/"
primary_forest_ds = xr.open_zarr(primary_forest_zarr_path, consolidated=False)
primary_forest_da = primary_forest_ds[cn.starting_composite_primary_forest_pattern]
if "year" in primary_forest_da.dims:
    primary_forest_da = primary_forest_da.isel(year=0, drop=True)
primary_forest_da = primary_forest_da.rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
primary_forest_labels = {0: "not primary forest", 1: "primary forest"}

# Resolution and dimensions of all inputs should be the same
print("land_state    x res:", float(land_state_da["x"][1] - land_state_da["x"][0]), " size:", land_state_da.sizes)
print("prim_for      x res:", float(primary_forest_ds["x"][1] - primary_forest_ds["x"][0]), " size:", primary_forest_ds.sizes)

land_state    x res: 0.0002499999999940883  size: Frozen({'year': 9, 'y': 720000, 'x': 1440000})
prim_for      x res: 0.0002499999999940883  size: Frozen({'x': 1440000, 'y': 720000, 'year': 1})


### Zonal stats

In [10]:
%%time

# Separate output objects for vegetation/LULUCF, mineral soil, and organic soil because they have
# different numbers of years (9 vs. 1 vs. 2)
annual_lazy = []
min_soil_lazy = []
organic_soil_lazy = []

print(f"Iterating through tiles to create task graph: {uu.timestr()}")

# Iterates through tiles to create tasks, but executes tasks from all tiles together in a single graph
for tile_id, subset in features_by_tile.items():

    # Tile's own true bounds, intersected with the bounds of just the features assigned to it
    tminx, tminy, tmaxx, tmaxy = uu.get_10x10_tile_bounds(tile_id)
    fminx, fminy, fmaxx, fmaxy = subset.total_bounds
    buffer = 0.001
    
    # Buffer is applied to the feature bounds BEFORE clamping to the tile's true edge, not after --
    # otherwise it can push the window past the tile boundary and overlap the neighboring tile,
    # double-counting any feature that crosses that boundary.
    minx = max(tminx, fminx - buffer)
    miny = max(tminy, fminy - buffer)
    maxx = min(tmaxx, fmaxx + buffer)
    maxy = min(tmaxy, fmaxy + buffer)

    ids = subset[field].astype("uint32").values
    shapes = list(zip(subset.geometry, ids))  # (geometry, zone_id) pairs to rasterize, small regardless of tile size
    unique_ids = np.unique(ids)

    # Builds a per-chunk rasterize function -- called lazily by dask.array.map_blocks below, so each
    # chunk's zone-ID raster is computed on a worker at compute time instead of eagerly on the client.
    # Only `shapes` and `transform` (both tiny) get shipped to workers, not a precomputed data array.
    # Needs to be inside the for loop so that each tile gets its own shapes and transform from these nested functions during lazy execution. 
    def make_rasterize_chunk_fn(shapes, transform):
        def rasterize_chunk(block, block_info=None):
            (y0, y1), (x0, x1) = block_info[None]["array-location"]
            chunk_transform = transform * Affine.translation(x0, y0)
            return rasterio.features.rasterize(
                shapes, out_shape=(y1 - y0, x1 - x0), transform=chunk_transform,
                fill=0, all_touched=False, dtype="uint32",
            )
        return rasterize_chunk

    # Pixel area zarr is in square meters -- converts to hectares to match the model's per-hectare density outputs
    pixel_area_bbox = pixel_area.sel(x=slice(minx, maxx), y=slice(maxy, miny))
    pixel_area_ha = pixel_area_bbox * cn.m2_to_ha

    # Contextual layers valid for the annual analysis layers (vegetation and LULUCF), sliced to this tile's window once, shared below
    land_state_bbox = land_state_da.sel(x=slice(minx, maxx), y=slice(maxy, miny))
    primary_forest_bbox = primary_forest_da.sel(x=slice(minx, maxx), y=slice(maxy, miny))

    # --- Vegetation + LULUCF totals (9 years), crossed with land_state_node and primary_forest ---
    annual_bboxes = [
        da_.sel(x=slice(minx, maxx), y=slice(maxy, miny)).assign_coords(analysis_layer=label)
        for label, da_ in annual_analysis_layers.items()
    ]
    veg_stacked = xr.concat(annual_bboxes, dim="analysis_layer").rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
    transform = veg_stacked.rio.transform()

    # Chunk-structure template only -- its values are never used, so no density data gets pulled in just to build the zone array
    zone_id_dask = da.map_blocks(
        make_rasterize_chunk_fn(shapes, transform),
        veg_stacked.isel(year=0, analysis_layer=0).data,
        dtype="uint32",
    )
    zone_da = xr.DataArray(zone_id_dask, dims=("y", "x"),
                           coords={"y": veg_stacked["y"], "x": veg_stacked["x"]}, name=field)

    # join="override" forces all five arrays onto the same coordinates rather than requiring exact
    # float equality -- avoids the alignment failures independently-computed coordinate arrays can cause
    annual_aligned, pixel_area_ha_v, zone_da_v, land_state_v, primary_forest_v = xr.align(
        veg_stacked, pixel_area_ha, zone_da, land_state_bbox, primary_forest_bbox, join="override"
    )
    annual_per_pixel = annual_aligned * pixel_area_ha_v

    # Groups by zone, land_state_node, and primary_forest simultaneously (crossed together), plus year --
    # analysis_layer is left out of the "by" args, so it's preserved as an output dimension rather than reduced
    veg_reduced = xarray_reduce(
        annual_per_pixel, zone_da_v, land_state_v, primary_forest_v, annual_per_pixel["year"],
        func="sum",
        expected_groups=(unique_ids, node_codes, cn.composite_primary_codes, annual_per_pixel["year"].values),
        fill_value=0,
        # reindex=ReindexStrategy(blockwise=False, array_type=ReindexArrayType.SPARSE_COO),  # Does not work with UNESCO WH site shp. Maybe data isn't sparse enough. Keeping here for other datasets. 
    )

    # --- Mineral soil (no year dim), crossed with primary_forest only -- land_state_node isn't valid here because it's annual ---
    min_soil_bboxes = [
        da_.sel(x=slice(minx, maxx), y=slice(maxy, miny)).assign_coords(analysis_layer=label)
        for label, da_ in min_soil_layers_static.items()
    ]
    min_soil_stacked = xr.concat(min_soil_bboxes, dim="analysis_layer").rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")

    # Reuses the SAME zone_da built above -- no need to rasterize the zones a second time
    min_soil_aligned, pixel_area_ha_s, zone_da_s, primary_forest_s = xr.align(
        min_soil_stacked, pixel_area_ha, zone_da, primary_forest_bbox, join="override"
    )
    min_soil_per_pixel = min_soil_aligned * pixel_area_ha_s

    # No year grouping here -- collapses straight to one value per zone per primary_forest class per mineral soil layer
    min_soil_reduced = xarray_reduce(
        min_soil_per_pixel, zone_da_s, primary_forest_s,
        func="sum", 
        expected_groups=(unique_ids, cn.composite_primary_codes), 
        fill_value=0,
        # reindex=ReindexStrategy(blockwise=False, array_type=ReindexArrayType.SPARSE_COO),  # Does not work with UNESCO WH site shp. Maybe data isn't sparse enough. Keeping here for other datasets. 
    )

    # --- Organic soil (2 native five-year intervals), crossed with primary_forest only -- land_state_node isn't valid here either ---
    organic_bboxes = [
        da_.sel(x=slice(minx, maxx), y=slice(maxy, miny)).assign_coords(analysis_layer=label)
        for label, da_ in organic_soil_layers_native.items()
    ]
    organic_stacked = xr.concat(organic_bboxes, dim="analysis_layer").rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")

    # Reuses the SAME zone_da built above -- no need to rasterize the zones a third time
    organic_aligned, pixel_area_ha_o, zone_da_o, primary_forest_o = xr.align(
        organic_stacked, pixel_area_ha, zone_da, primary_forest_bbox, join="override"
    )
    organic_per_pixel = organic_aligned * pixel_area_ha_o

    # Groups by zone and primary_forest, plus organic soil's own native year dimension (2 intervals: 2016_2020, 2021_2024)
    organic_reduced = xarray_reduce(
        organic_per_pixel, zone_da_o, primary_forest_o, organic_per_pixel["year"],
        func="sum",
        expected_groups=(unique_ids, cn.composite_primary_codes, organic_per_pixel["year"].values),
        fill_value=0,
        # reindex=ReindexStrategy(blockwise=False, array_type=ReindexArrayType.SPARSE_COO),  # Results in indefinite task retries with UNESCO WH site shp. Maybe data isn't sparse enough. Keeping here for other datasets. 
    )

    # Appends the still-lazy (uncomputed) reduce objects -- nothing has touched S3 or actually summed
    # anything yet at this point in the loop
    annual_lazy.append((tile_id, veg_reduced))
    min_soil_lazy.append((tile_id, min_soil_reduced))
    organic_soil_lazy.append((tile_id, organic_reduced))

print(f"Created task graph: {uu.timestr()}")

full_graph = dask.base.collections_to_dsk([r for _, r in annual_lazy] + [r for _, r in min_soil_lazy] + [r for _, r in organic_soil_lazy])
print("Number of tasks in the graph:", len(full_graph))

Iterating through tiles to create task graph: 20260909_20_44_20
Created task graph: 20260909_20_46_26
Number of tasks in the graph: 164992
CPU times: user 24 s, sys: 4.94 s, total: 28.9 s
Wall time: 2min 15s


In [15]:
# Creates the Coiled cluster just before it's needed, so it's not sitting idle during data preparation

cluster = coiled.Cluster( 
    n_workers=50,  
    use_best_zone=True, 
    compute_purchase_option="spot_with_fallback",
    idle_timeout="15 minutes",
    region="us-east-1", # close to dataset, avoid egress charges
    name="custom_AOI_zonal_stats",
    workspace='wri-land-research',
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r8g.2xlarge",   # 64 GB
    worker_vm_types="r8g.2xlarge",      # 64 GB
    **({'software': 'afolu_zonal_stats_20251222'}) 
)

client = cluster.get_client()

Output()

Could not get token from client GCP session. This is not a concern unless you're planning to use forwarded GCP credentials on your cluster. The error was: Reauthentication is needed. Please run `gcloud auth application-default login` to reauthenticate.
/home/melrose94/miniforge3/envs/afolu_backup/lib/python3.13/site-packages/distributed/client.py:1612: VersionMismatchWarning: Mismatched versions found

+-------------+----------------+-----------------+-----------------+
| Package     | Client         | Scheduler       | Workers         |
+-------------+----------------+-----------------+-----------------+
| cloudpickle | 3.1.1          | 3.1.2           | 3.1.2           |
| msgpack     | 1.1.0          | 1.2.1           | 1.2.1           |
| pandas      | 2.3.0          | 2.2.3           | 2.2.3           |
| python      | 3.13.3.final.0 | 3.12.11.final.0 | 3.12.11.final.0 |
| toolz       | 1.0.0          | 1.1.0           | 1.1.0           |
| tornado     | 6.5.1          | 6.5.7    

In [16]:
%%time

### Actual execution

# One combined compute() call still lets the scheduler parallelize everything at once, across every
# tile's chunks together, rather than finishing one tile before starting the next
print(f"Computing: {uu.timestr()}")
computed = dask.compute(
    *[r for _, r in annual_lazy],
    *[r for _, r in min_soil_lazy],
    *[r for _, r in organic_soil_lazy],
)
computed_veg = computed[:len(annual_lazy)]
computed_min_soil = computed[len(annual_lazy):len(annual_lazy) + len(min_soil_lazy)]
computed_organic = computed[len(annual_lazy) + len(min_soil_lazy):]
print(f"Done computing: {uu.timestr()}")

cluster.shutdown()

Computing: 20260909_21_20_40


/home/melrose94/miniforge3/envs/afolu_backup/lib/python3.13/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 146.22 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


KilledWorker: Attempted to run task ('getitem-open_dataset-broadcast_to-3bec7c7525949d371bff871ab259f96c', 0, 0, 2, 0) on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tls://10.0.233.205:45747. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [17]:
%%time

# --- Vegetation + LULUCF, broken down by land_state -- mineral soil and organic soil columns are
# zeroed here since neither is broken down by land_state; their real values go on a shared
# placeholder row below (both sources on the same row, not one each).
print("Creating table with annual datasets (vegetation and LULUCF)")
veg_dfs = []
for (tile_id, _), result in zip(annual_lazy, computed_veg):
    df = result.to_dataframe(name="value").reset_index()
    df["year"] = df["year"].map(lambda i: years[i])
    veg_dfs.append(df)
veg_combined = pd.concat(veg_dfs, ignore_index=True)
veg_combined = veg_combined.groupby(
    # [field, "year", cn.land_state_pattern, cn.starting_composite_primary_forest_pattern, "analysis_layer"],   # v1.0.6
    [field, "year", "land_state_node", cn.starting_composite_primary_forest_pattern, "analysis_layer"],   #v1.0.5
    as_index=False,
)["value"].sum()
veg_combined = veg_combined[veg_combined["value"] != 0]

# if cn.land_state_pattern in veg_combined.columns:   # v1.0.6
if "land_state_node" in veg_combined.columns:   #v1.0.5
    veg_combined = veg_combined.merge(
        state_node_df[['land_state', 'land_state_meaning', 'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type', 'tree_focused_2x2']],
        # left_on=cn.land_state_pattern, right_on="land_state", how="left",   # v1.0.6
        left_on="land_state_node", right_on="land_state", how="left",   # v1.0.5
    ).drop(columns=["land_state"])

if cn.starting_composite_primary_forest_pattern in veg_combined.columns:
    veg_combined[cn.starting_composite_primary_forest_pattern] = veg_combined[cn.starting_composite_primary_forest_pattern].map(primary_forest_labels)

display(veg_combined)

veg_wide = veg_combined.pivot_table(
    # index=[field, "year", cn.land_state_pattern, "land_state_meaning", "land_state_broad_class", "land_state_detailed_class", "tall_veg_type", "tree_focused_2x2", cn.starting_composite_primary_forest_pattern],   # v1.0.6
    index=[field, "year", "land_state_node", "land_state_meaning", "land_state_broad_class", "land_state_detailed_class", "tall_veg_type", "tree_focused_2x2", cn.starting_composite_primary_forest_pattern],   #v1.0.5
    columns="analysis_layer", values="value", fill_value=0,
).reset_index()
for col in min_soil_var_names:
    veg_wide[col] = 0
for col in organic_soil_var_names:
    veg_wide[col] = 0

# --- Mineral soil, broken down by primary_forest only -- no land_state or other-source columns added
# yet; those get added once, after merging with organic soil below, so both land on the same rows.
print("Creating mineral soil table")
min_soil_dfs = [result.to_dataframe(name="value").reset_index() for _, result in zip(min_soil_lazy, computed_min_soil)]
min_soil_combined = pd.concat(min_soil_dfs, ignore_index=True)
min_soil_combined = min_soil_combined.groupby(
    [field, cn.starting_composite_primary_forest_pattern, "analysis_layer"], as_index=False
)["value"].sum()

min_soil_combined[cn.starting_composite_primary_forest_pattern] = min_soil_combined[cn.starting_composite_primary_forest_pattern].map(primary_forest_labels)
min_soil_wide = min_soil_combined.pivot_table(
    index=[field, cn.starting_composite_primary_forest_pattern], columns="analysis_layer", values="value", fill_value=0,
).reset_index()
min_soil_wide = min_soil_wide.merge(pd.DataFrame({"year": years}), how="cross")

# --- Organic soil, broken down by primary_forest only, its 2 native five-year intervals broadcast
# onto the 9 annual years (same interval's value repeats across every year it spans)
print("Creating organic soil table")
organic_dfs = [result.to_dataframe(name="value").reset_index() for _, result in zip(organic_soil_lazy, computed_organic)]
organic_combined = pd.concat(organic_dfs, ignore_index=True)
organic_combined = organic_combined.groupby(
    [field, "year", cn.starting_composite_primary_forest_pattern, "analysis_layer"], as_index=False
)["value"].sum()

organic_combined[cn.starting_composite_primary_forest_pattern] = organic_combined[cn.starting_composite_primary_forest_pattern].map(primary_forest_labels)
organic_wide = organic_combined.pivot_table(
    index=[field, "year", cn.starting_composite_primary_forest_pattern], columns="analysis_layer", values="value", fill_value=0,
).reset_index()

def organic_soil_interval_idx(year):
    for i, interval in enumerate(cn.organic_soil_year_intervals):
        start, end = (int(y) for y in interval.split("_"))
        if start <= year <= end:
            return i
    raise ValueError(f"No organic soil interval covers year {year}")

year_to_interval_idx = pd.DataFrame({"year": years, "interval_idx": [organic_soil_interval_idx(y) for y in years]})
organic_wide = organic_wide.rename(columns={"year": "interval_idx"}).merge(
    year_to_interval_idx, on="interval_idx", how="left"
).drop(columns="interval_idx")

# --- Merges mineral soil and organic soil into ONE combined soil table, matched on field/year/
# primary_forest -- this is what actually puts them on the same row, not just a matching label string.
print("Combining mineral and organic soil tables")
soil_wide = min_soil_wide.merge(organic_wide, on=[field, "year", cn.starting_composite_primary_forest_pattern], how="outer")
# soil_wide[cn.land_state_pattern] = 0    # v1.0.6
soil_wide["land_state_node"] = 0   # v1.0.5
soil_wide["land_state_meaning"] = "no_land_state__soil"
soil_wide["land_state_broad_class"] = "no_land_state__soil"
soil_wide["land_state_detailed_class"] = "no_land_state__soil"
soil_wide["tall_veg_type"] = "no_land_state__soil"
soil_wide["tree_focused_2x2"] = "no_land_state__soil"
for col in list(veg_var_names.keys()) + list(lulucf_var_names.keys()):
    soil_wide[col] = 0

# --- Combine via concat (not merge) -- each source's total appears exactly once in the stacked table
print("Combining tables")
wide_combined = pd.concat([veg_wide, soil_wide], ignore_index=True)
wide_combined = wide_combined.merge(gdf[[field] + output_fields].astype({field: "uint32"}), on=field, how="left")
wide_combined["organic_soil_emissions_total_MgCO2e"] = wide_combined["organic_soil_emissions_fire_MgCO2e"] + wide_combined["organic_soil_emissions_drainage_MgCO2e"]
wide_combined = wide_combined[
    [field] + output_fields
    # + ["year", cn.land_state_pattern, "land_state_meaning", "land_state_broad_class", "land_state_detailed_class", "tall_veg_type", "tree_focused_2x2", cn.starting_composite_primary_forest_pattern] # v1.0.6
    + ["year", "land_state_node", "land_state_meaning", "land_state_broad_class", "land_state_detailed_class", "tall_veg_type", "tree_focused_2x2", cn.starting_composite_primary_forest_pattern]   # v1.0.5
    + list(var_names.keys()) + ["organic_soil_emissions_total_MgCO2e"]
]

print(f"Rows in dataframe: {len(wide_combined)}")
display(wide_combined)

Creating table with annual datasets (vegetation and LULUCF)


NameError: name 'computed_veg' is not defined

In [18]:
%%time

# Saves to Excel
print("Saving to Excel")
readme_df = pd.DataFrame({"": readme_lines})

xlsx_path = f'/mnt/c/GIS/{out_name}_{today_sparse}.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    readme_df.to_excel(writer, sheet_name="readme", index=False, header=False)
    wide_combined.to_excel(writer, sheet_name="data", index=False)
print("Saved to Excel")

Saving to Excel


NameError: name 'wide_combined' is not defined